# CASMI Research 0004_generation_shared
Development-gated experimental mass hypotheses; repeated development validation, no independent acceptance for this extension.


In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, os, shutil, subprocess, sys, tarfile, zipfile
archives = list(Path('/kaggle/input').rglob('casmi_chemistry_bundle.zip'))
bundle = Path('/kaggle/temp/casmi_chemistry_runtime')
bundle.mkdir(parents=True, exist_ok=True)
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(bundle)
else:
    roots = list(Path('/kaggle/input').rglob('deployment_recipe.json'))
    assert len(roots) == 1, roots
    shutil.copytree(roots[0].parent, bundle, dirs_exist_ok=True)
for name, expected in json.loads((bundle / 'SHA256SUMS.json').read_text()).items():
    assert hashlib.sha256((bundle / name).read_bytes()).hexdigest() == expected, name
required = json.loads((bundle / 'runtime_versions.json').read_text())
try:
    installed_rdkit = importlib.metadata.version('rdkit')
except importlib.metadata.PackageNotFoundError:
    installed_rdkit = None
if installed_rdkit != required['rdkit']:
    wheels = list((bundle / 'wheels').glob('rdkit*.whl'))
    assert len(wheels) == 1
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', str(wheels[0])])
java_root = bundle / 'java'
java_root.mkdir(exist_ok=True)
with tarfile.open(bundle / 'java-runtime.bin') as archive:
    archive.extractall(java_root, filter='data')
java = list(java_root.glob('*/bin/java'))
assert len(java) == 1
java[0].chmod(0o755)
os.environ['PATH'] = str(java[0].parent) + os.pathsep + os.environ['PATH']
os.environ['PYTHONPATH'] = str(bundle)
subprocess.check_call(['java', '-version'])
smoke = """from casmi_ml.metfrag import MetFrag
from casmi_ml.chemistry import composition_mass
row = {'precursor_mz': composition_mass('C2H6O') + 1.007276466621,
       'adduct': '[M+H]+', 'instrument_type': 'QTOF',
       'ms2_mzs': [31.01839, 29.038576], 'ms2_normalized_intensities': [1.0, 0.6]}
result = MetFrag('metfrag.jar', 'smoke-cache').score(row, {'ethanol': 'CCO', 'ether': 'COC'})
assert result['status'] == 'complete', result
assert set(result['scores']) == {'ethanol', 'ether'}, result
print('MetFrag smoke:', result)
"""
subprocess.check_call([sys.executable, '-c', smoke], cwd=bundle)
subprocess.check_call([sys.executable, '-m', 'casmi_ml.research_pipeline',
    '--recipe', str(bundle / 'deployment_recipe.json'),
    '--data-dir', '/kaggle/input/competition',
    '--coconut', '/kaggle/input/coconut_structures.parquet',
    '--output', '/kaggle/working/submission.csv'], cwd=bundle)
report = json.loads(Path('/kaggle/working/submission.csv.report.json').read_text())
assert report['seconds'] < 1800, report
assert report['peak_rss_mib'] < 8192, report
print(report)
